# Build the complete cell — all three models, all the data
This notebook genuinely **runs all three models** and **fetches every data source**, writes what each model
produces into the cell, then assembles and serves the interactive result.

| model | runs here | data it fetches | file it writes | how the cell uses it |
|---|---|---|---|---|
| **1. our integrated model** | trains a classifier on our 14-layer features + DepMap truth | DepMap CRISPRGeneEffect, repo features | `predicted_essentiality.csv` | fills essentiality for genes with no CRISPR label |
| **2. atlas / scGPT cell-type** | per-cell-type expression from the human atlas | CELLxGENE census (Tabula Sapiens) | `celltype_masters.json`, `celltype_expression.csv` | data-driven master TFs → cell-type active networks |
| **3. Geneformer** | in-silico *delete* perturbation | Geneformer weights (HF) + tokenized atlas cells | `gf_perturb.json` | enriches the knockout cascade with predicted downstream genes |
| **4. perturbation-response predictor** | trains gene features → Perturb-seq KO-signature space, held-out validated (recall@10 vs random) | Perturb-seq (Replogle 2022) + our 34-layer features | `model4_predictions.json` | predicted functional neighbors + affected pathway for the ~14k never-perturbed genes, tagged *predicted, not measured* |

**Runtime:** set *Runtime → A100* (Model 2 census + Model 3 Geneformer want a GPU and ~15–40 min).
Each model cell is guarded — if one environment isn't available the notebook still completes and builds a cell,
but on a proper GPU+internet runtime **all three run for real.** See `docs/CELL_ARCHITECTURE.md`.


In [ ]:
# [1] repo (scripts + pre-processed network/compartment/HIV data) 
import os, sys, json
if not os.path.exists('colab/build_cell_complete.py'):
    !git clone -q --branch claude/vectorize-gex-propensity-NRqBW https://github.com/nikku03/cell.git
    os.chdir('cell')
print('cwd:', os.getcwd())
IN_COLAB='google.colab' in sys.modules


In [ ]:
# [2] installs
!pip -q install torch scikit-learn pandas numpy scipy anndata huggingface_hub
!pip -q install cellxgene-census            # Model 2 (atlas)
OUT='outputs/orphan'; os.makedirs(OUT, exist_ok=True)


In [ ]:
# [2b] fetch the raw data build_cell_complete.py needs (the big files are NOT in the repo).
#      idempotent: skips anything already present. These are all the external sources the cell uses.
import urllib.request, gzip, shutil
H='data/external_data/human'; os.makedirs(H, exist_ok=True)
SRC={
 'gene_info.gz':'https://ftp.ncbi.nlm.nih.gov/gene/DATA/GENE_INFO/Mammalia/Homo_sapiens.gene_info.gz',
 'string_aliases.txt.gz':'https://stringdb-downloads.org/download/protein.aliases.v12.0/9606.protein.aliases.v12.0.txt.gz',
 'string_physical.txt.gz':'https://stringdb-downloads.org/download/protein.physical.links.v12.0/9606.protein.physical.links.v12.0.txt.gz',
 'human_gem.txt':'https://raw.githubusercontent.com/SysBioChalmers/Human-GEM/main/model/Human-GEM.txt',   # genome-scale metabolism
 'signor.tsv':'https://signor.uniroma2.it/getData.php?organism=9606',                                     # signaling
 'complexportal.tsv':'https://ftp.ebi.ac.uk/pub/databases/intact/complex/current/complextab/9606.tsv',    # named complexes
 'cellphonedb.csv':'https://raw.githubusercontent.com/ventolab/cellphonedb-data/master/data/interaction_input.csv',  # ligand-receptor
 'dgidb.tsv':'https://dgidb.org/data/latest/interactions.tsv',                                            # drug-gene
 'uniprot_acc_ptm.tsv':'https://rest.uniprot.org/uniprotkb/stream?query=organism_id:9606+AND+reviewed:true&fields=accession,gene_primary,ft_mod_res&format=tsv',  # PTMs + acc->sym
 'bioplex.tsv':'https://bioplex.hms.harvard.edu/data/BioPlex_293T_Network_10K_Dec_2019.tsv',           # AP-MS interactome
 'opencell.csv':'https://opencell.czbiohub.org/data/datasets/opencell-protein-interactions.csv',       # proximity-labeling interactome
 'huri.tsv':'https://interactome-atlas.org/data/HuRI.tsv',                                             # HuRI systematic Y2H binary interactome (ENSG-ENSG); note: bare host (www. has a bad cert)
}
def _dl(url,p):
    try:
        urllib.request.urlretrieve(url,p)
        if os.path.getsize(p) < 1024: raise IOError('empty/tiny response (%d bytes) - likely a WAF/redirect challenge' % os.path.getsize(p))
        print('   ->',os.path.getsize(p)//1024,'KB'); return True
    except Exception as e:
        print('   !! skip',os.path.basename(p),'- download failed:',repr(e)[:120])
        try: os.remove(p)
        except: pass
        return False
OPTIONAL={'huri.tsv'}   # nice-to-have sources whose absence the build tolerates
for fn,url in SRC.items():
    p=f'{H}/{fn}'
    if os.path.exists(p) and os.path.getsize(p)>10000: print('have',fn); continue
    print('downloading',fn,'...'); ok=_dl(url,p)
    assert ok or fn in OPTIONAL, fn+' failed to download and is required'
hp=f'{H}/hiv_interactions'   # NCBI GeneRIF, gunzip to the plain file the builder reads
if not (os.path.exists(hp) and os.path.getsize(hp)>10000):
    print('downloading hiv_interactions ...')
    urllib.request.urlretrieve('https://ftp.ncbi.nlm.nih.gov/gene/GeneRIF/hiv_interactions.gz',hp+'.gz')
    with gzip.open(hp+'.gz','rb') as f,open(hp,'wb') as o: shutil.copyfileobj(f,o)
# DepMap CRISPR (~380 MB) -> MEASURED essentiality (overrides predictions)
dmp=f'{H}/CRISPRGeneEffect.csv'
if not (os.path.exists(dmp) and os.path.getsize(dmp)>1e7):
    print('downloading DepMap CRISPRGeneEffect (~380 MB) ...'); _dl('https://ndownloader.figshare.com/files/43346616', dmp)
import subprocess as _sp; _sp.run([sys.executable,'colab/compute_depmap_essentiality.py'],check=True)  # -> depmap_essentiality.csv
_sp.run([sys.executable,'colab/compute_depmap_codep.py'],check=True)  # -> depmap_codep.json + depmap_sl.json (co-essentiality + SL)
# Perturb-seq (Replogle 2022 genome-wide K562 pseudobulk, ~375 MB) -> MEASURED functional neighbors for dark genes.
# One transcriptional signature per perturbed gene; correlating signatures gives function by guilt-by-association.
WITH_PERTURBSEQ=os.environ.get('WITH_PERTURBSEQ','1')=='1'   # set 0 to skip the 375 MB download
if WITH_PERTURBSEQ:
    psh=f'{H}/perturbseq_gwps_bulk.h5ad'
    if not (os.path.exists(psh) and os.path.getsize(psh)>1e7):
        print('downloading Perturb-seq genome-wide pseudobulk (~357 MB) ...'); _dl(os.environ.get('PERTURBSEQ_URL','https://ndownloader.figshare.com/files/35773217'), psh)   # ndownloader host (plus.figshare is WAF-blocked)
    # OPTIONAL extra cell lines -> broadens perturbation coverage (compute_perturbseq merges every perturbseq_*.h5ad).
    # Set these env vars to a direct h5ad link (e.g. scPerturb Zenodo) to include them; left blank they are skipped.
    # RPE1 (2nd cell line, ~90 MB) has a working default; Norman combinatorial is env-only.
    for fn,env,default in [('perturbseq_rpe1_bulk.h5ad','PERTURBSEQ_RPE1_URL','https://ndownloader.figshare.com/files/35775512'),('perturbseq_norman.h5ad','PERTURBSEQ_NORMAN_URL','')]:
        u=os.environ.get(env,default)
        if u and not (os.path.exists(f'{H}/{fn}') and os.path.getsize(f'{H}/{fn}')>1e7):
            print('downloading',fn,'...'); _dl(u, f'{H}/{fn}')
    _sp.run([sys.executable,'colab/compute_perturbseq.py'],check=True)  # -> perturbseq_neighbors.json (merged across all datasets)
else: print('WITH_PERTURBSEQ=0 -> dark-gene function falls back to co-essentiality + PPI neighbors only')
# === Tier-1/2 gap-filling datasets (each processor skips gracefully if its file is absent) ===
WITH_EXTRA=os.environ.get('WITH_EXTRA','1')=='1'   # set 0 to skip all four heavy adds below
def _get(fn,url,minkb=100):
    p=f'{H}/{fn}'
    if url and not (os.path.exists(p) and os.path.getsize(p)>minkb*1024):
        try:
            print('downloading',fn,'...'); urllib.request.urlretrieve(url,p); print('   ->',os.path.getsize(p)//1024,'KB')
        except Exception as e:
            print('   !! skip',fn,'- download failed:',repr(e)[:120]); return False   # never crash the run on a bad URL
    return os.path.exists(p)
if WITH_EXTRA:
    # refGene TSS (shared by ReMap + GTEx)
    _get('refGene.txt.gz','https://hgdownload.soe.ucsc.edu/goldenPath/hg38/database/refGene.txt.gz')
    # (1) ReMap 2022 non-redundant TF ChIP-seq peaks -> TF->target edges
    _get('remap.bed.gz', os.environ.get('REMAP_URL','https://remap.univ-amu.fr/storage/remap2022/hg38/MACS2/remap2022_nr_macs2_hg38_v1_0.bed.gz'), minkb=1000)
    _sp.run([sys.executable,'colab/compute_remap.py'])                      # -> remap_tf_targets.tsv
    # (2) GTEx v8 trans-eQTLs -> cross-gene regulatory candidates (set GTEX_TRANS_URL to a direct link)
    _get('gtex_trans_pairs.txt', os.environ.get('GTEX_TRANS_URL','https://storage.googleapis.com/adult-gtex/bulk-qtl/v8/single-tissue-trans-qtl/GTEx_Analysis_v8_trans_eGenes_fdr05.txt'))
    _sp.run([sys.executable,'colab/compute_gtex_eqtl.py'])                  # -> gtex_trans_edges.tsv
    # (3) LINCS L1000 Level-5 signatures -> perturbation neighbors (set LINCS_GCTX_URL + LINCS_SIGINFO_URL; large)
    _get('lincs_level5.gctx', os.environ.get('LINCS_GCTX_URL',''), minkb=10000)
    _get('lincs_siginfo.txt.gz', os.environ.get('LINCS_SIGINFO_URL',''))
    _sp.run([sys.executable,'colab/compute_lincs.py'])                      # -> lincs_neighbors.json
    # (4) NicheNet ligand->target matrix (R .rds) -> tissue-model downstream wiring
    !pip -q install pyreadr
    _get('nichenet_ligand_target_matrix.rds', os.environ.get('NICHENET_URL','https://zenodo.org/record/7074291/files/ligand_target_matrix_nsga2r_final.rds'), minkb=1000)
    _sp.run([sys.executable,'colab/compute_nichenet.py'])                   # -> nichenet_ligand_targets.json (used by build_tissue_model)
# (5) Phase-1 ARCHS4 co-expression network -> the 8th independent lens (reads the 59 GB h5 in place from Drive)
WITH_COEXPR=os.environ.get('WITH_COEXPR','1')=='1'
if WITH_COEXPR:
    !pip -q install h5py
    _sp.run([sys.executable,'colab/compute_coexpr.py'])                     # -> coexpr_neighbors.json (auto-finds ARCHS4 in expression_geo/)
# Phase-3 causal regulome: ReMap binding x Perturb-seq response -> signed causal edges (needs both above)
_sp.run([sys.executable,'colab/compute_causal_reg.py'])                     # -> causal_reg.tsv + causal_reg.json
# (6) Phase-5 ncRNA -> target layer (miRTarBase + LncTarD; URLs are release-specific, pass via env)
if WITH_EXTRA:
    _get('mirtarbase_hsa.xlsx', os.environ.get('MIRTARBASE_URL',''))
    _get('lnctard.txt', os.environ.get('LNCTARD_URL','https://lnctard.bio-database.com/download'))
    !pip -q install openpyxl
    _sp.run([sys.executable,'colab/compute_ncrna.py'])                      # -> ncrna_targets.json
lp=f'{H}/hiccups_loops.txt.gz'   # 3D chromatin loops (GM12878 HiCCUPS)
if not (os.path.exists(lp) and os.path.getsize(lp)>1000):
    urllib.request.urlretrieve('https://ftp.ncbi.nlm.nih.gov/geo/series/GSE63nnn/GSE63525/suppl/GSE63525_GM12878_primary%2Breplicate_HiCCUPS_looplist.txt.gz', lp)
g2p=f'{H}/gene2pubmed.gz'   # literature coverage -> Target Intelligence white-space
if not (os.path.exists(g2p) and os.path.getsize(g2p)>1e6):
    urllib.request.urlretrieve('https://ftp.ncbi.nlm.nih.gov/gene/DATA/gene2pubmed.gz', g2p)
_sp.run([sys.executable,'colab/compute_literature.py'],check=True)      # -> literature_counts.json
_sp.run([sys.executable,'colab/target_intelligence.py'],check=True)     # -> ti_*.json (target-priority, white-space, combos)
# CCLE omics -> biomarkers of sensitivity (#4/#5)
for fn,fid in [('CCLE_expression.csv','46490878'),('CCLE_mutations_damaging.csv','46500376')]:
    pth=f'{H}/{fn}'
    if not (os.path.exists(pth) and os.path.getsize(pth)>1e7):
        print('downloading',fn,'...'); urllib.request.urlretrieve(f'https://ndownloader.figshare.com/files/{fid}', pth)
_sp.run([sys.executable,'colab/compute_biomarkers.py'],check=True)      # -> biomarkers.json
_sp.run([sys.executable,'colab/compute_metabolic_graph.py'],check=True) # -> metabolic_graph.json
_sp.run([sys.executable,'colab/build_metabolic_routes.py'],check=True)  # -> metabolic_routes.html (alternative pathways)
for fn in ['gene_compartment.json','collectri.tsv','reactome_human.txt','gene_info.gz','string_aliases.txt.gz','string_physical.txt.gz','hiv_interactions','human_gem.txt','signor.tsv','complexportal.tsv','cellphonedb.csv','dgidb.tsv','uniprot_acc_ptm.tsv']:
    ok=os.path.exists(f'{H}/{fn}'); print(('OK  ' if ok else 'MISSING  ')+fn)
    assert ok, fn+' missing — build will fail'


In [ ]:
# [3] (optional) persist to Drive
PROJ=None
if IN_COLAB:
    try:
        from google.colab import drive; drive.mount('/content/drive')
        PROJ='/content/drive/MyDrive/cell_model'; os.makedirs(PROJ, exist_ok=True)
    except Exception as e: print('no Drive:', e)
print('persist ->', PROJ or 'local only')


## Model 1 — our integrated essentiality model
Trains on our 14-layer features with **measured DepMap essentiality** as truth, then predicts the unlabeled genes. Baseline to match: features-only AUC ~0.97. Output → `predicted_essentiality.csv`.


In [ ]:
import pandas as pd, numpy as np
bb=pd.read_csv(f'{OUT}/integrated_cell_human.csv')
print('backbone genes:', len(bb))
# --- fetch DepMap measured essentiality (truth). Update the figshare file id from depmap.org/portal/download ---
DEPMAP_URL=os.environ.get('DEPMAP_URL','')  # e.g. a CRISPRGeneEffect.csv figshare direct link
dep=None
if DEPMAP_URL:
    try:
        d=pd.read_csv(DEPMAP_URL, index_col=0)      # cell lines x genes (Chronos)
        frac_dep=(d< -0.5).mean(axis=0)             # fraction of lines where gene is essential
        frac_dep.index=[g.split(' ')[0] for g in frac_dep.index]
        dep=pd.Series(frac_dep.groupby(level=0).max())
        print('DepMap genes:', len(dep))
    except Exception as e: print('DepMap fetch failed, using repo labels:', e)
else: print('set DEPMAP_URL env for extra truth; using repo essential labels')


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score
FEATS=['loeuf','is_tf','regulon_out','regulators_in','ppi_degree','n_pathways','cpg_promoter','enhancers','n_diseases']
X=bb.copy()
for c in ['regulon_out','regulators_in','ppi_degree','n_pathways','enhancers','n_diseases']: X[c]=np.log1p(X[c].fillna(0))
X['loeuf']=X['loeuf'].fillna(X['loeuf'].median()); X[FEATS]=X[FEATS].fillna(0)
y=bb['essential'].copy()
if dep is not None:                              # merge DepMap truth where we have it
    m=bb['gene'].map(lambda g: dep.get(g, np.nan))
    y=y.where(y.notna(), (m>0.5).astype(float).where(m.notna()))
lab=y.notna().values
clf=GradientBoostingClassifier(max_depth=3,n_estimators=300,learning_rate=0.05)
cvp=cross_val_predict(clf,X.loc[lab,FEATS],y[lab],cv=5,method='predict_proba')[:,1]
print('Model 1 CV AUC:', round(roc_auc_score(y[lab],cvp),3),'| labeled:',int(lab.sum()))
clf.fit(X.loc[lab,FEATS],y[lab])
prob=clf.predict_proba(X[FEATS])[:,1]
pred=pd.DataFrame({'gene':bb['gene'],'pred':(prob>0.5).astype(int),'prob':prob.round(3)})
pred=pred[y.isna().values]                       # only the genes with NO measured label
pred.to_csv(f'{OUT}/predicted_essentiality.csv',index=False)
print('Model 1 -> predicted_essentiality.csv for', len(pred),'unlabeled genes;', int(pred.pred.sum()),'predicted essential')


## Model 2 — atlas / scGPT cell-type layer (Tabula Sapiens via CELLxGENE census)
Pulls per-cell-type mean expression across the human atlas, then for each cell type finds its **master TFs** = transcription factors most *specific* to that type (mean-expr in type ÷ mean across types). Subsamples cells first to avoid OOM. Output → `celltype_masters.json`, `celltype_expression.csv`.


In [ ]:
import cellxgene_census, scipy.sparse as sp, traceback
TFset=set(bb.loc[bb.is_tf==1,'gene'])
adata=None; cell_means=None
TISSUES=['blood','liver','heart','lung','brain','kidney']  # tissue_general controlled vocab
N_TYPES=40; PER_TYPE=60; MIN_CELLS=25   # abundant cell types, few cells each (all genes -> keep cells low)
try:
    with cellxgene_census.open_soma(census_version='2025-11-08') as census:
        VF="is_primary_data==True and tissue_general in "+repr(TISSUES)
        # 1) metadata only (cheap) -> pick abundant cell types, subsample their cells
        obs=pd.DataFrame(cellxgene_census.get_obs(census,'Homo sapiens',value_filter=VF,
                                     column_names=['soma_joinid','cell_type']))
        vc=obs['cell_type'].value_counts()
        keep=vc[vc>=MIN_CELLS].head(N_TYPES).index          # 40 most common cell types
        obs=obs[obs['cell_type'].isin(keep)]
        ids=(obs.groupby('cell_type',observed=True,group_keys=False)
                .apply(lambda d:d.sample(min(len(d),PER_TYPE),random_state=0)))
        print('fetching',len(ids),'cells x ALL genes across',ids['cell_type'].nunique(),'cell types...')
        # 2) fetch ALL genes (needed for abundance + cell-type wiring + differentiation);
        #    cells kept low (60/type) so the pull stays fast
        adata=cellxgene_census.get_anndata(census,'Homo sapiens',
               obs_coords=sorted(ids['soma_joinid'].tolist()),
               obs_column_names=['cell_type'])
    print('got',adata.shape[0],'cells x',adata.shape[1],'genes')
    # per-cell-type mean of CP10k-normalized expression, computed SPARSELY
    X=adata.X.tocsr() if sp.issparse(adata.X) else sp.csr_matrix(adata.X)
    lib=np.asarray(X.sum(1)).ravel(); lib[lib==0]=1
    Xn=X.multiply(1e4/lib[:,None]).tocsr()
    cts=adata.obs['cell_type'].astype(str).values
    uc=pd.unique(cts); ci={c:i for i,c in enumerate(uc)}
    rows=np.array([ci[c] for c in cts])
    Ind=sp.csr_matrix((np.ones(len(rows)),(rows,np.arange(len(rows)))),shape=(len(uc),len(rows)))
    counts=np.asarray(Ind.sum(1)).ravel()
    means=np.log1p(np.asarray((Ind@Xn).todense())/counts[:,None])
    genes=adata.var['feature_name'].astype(str).values
    cell_means=pd.DataFrame(means,index=uc,columns=genes)
    cell_means=cell_means.loc[:,~cell_means.columns.duplicated()]
    cell_means.to_csv(f'{OUT}/celltype_expression.csv')
    print('cell-type expression:',cell_means.shape)
except Exception:
    traceback.print_exc(); print('Model 2 census unavailable -> curated masters fallback')


In [ ]:
masters={}
if cell_means is not None:
    glob=cell_means.mean(axis=0)+1e-6
    spec=cell_means.div(glob,axis=1)              # specificity per cell type
    for ct in cell_means.index:
        tfs=[g for g in spec.columns if g in TFset]
        top=spec.loc[ct,tfs].sort_values(ascending=False).head(4)
        top=[g for g in top.index if cell_means.loc[ct,g]>0.5]   # expressed & specific
        if top: masters[str(ct)]=top
    masters=dict(list(masters.items())[:40])
    json.dump(masters,open(f'{OUT}/celltype_masters.json','w'),indent=1)
    print('Model 2 -> celltype_masters.json:',len(masters),'cell types')
    for k,v in list(masters.items())[:8]: print('  ',k,'->',v)
else: print('Model 2: no celltype_masters.json (builder uses validated curated masters)')


## Model 3 — Geneformer perturbation neighbors
For each key regulator, we ask Geneformer which genes are **functionally closest in its learned space** — the genes most likely to move when that regulator is perturbed. We use Geneformer's pretrained **gene embeddings** (robust, runs on CPU or GPU) as the primary signal; the heavier `InSilicoPerturber` (true in-silico *delete*) is included below as an optional upgrade. Output → `gf_perturb.json`.


In [ ]:
# target regulators = curated hubs + the atlas-derived master TFs from Model 2
TARGETS=sorted(set(['POLR2A','TP53','MYC','GATA4','HNF4A','SPI1','STAT1','EGFR','CTNNB1','RB1','JUN','FOXP3',
        'GATA3','TCF7','EOMES','CEBPB','NKX2-5','TBX5','ERG','FLI1','HNF1A','FOXA2','NEUROD1']
        +[g for v in (masters.values() if 'masters' in dir() and masters else []) for g in v]))
gf_out={}; gf_src='none'
# ensembl<->symbol for ALL genes (cheap var-only census read)
import cellxgene_census
with cellxgene_census.open_soma(census_version='2025-11-08') as _cx:
    _var=pd.DataFrame(cellxgene_census.get_var(_cx,'Homo sapiens',column_names=['feature_id','feature_name']))
ens2sym=dict(zip(_var.feature_id,_var.feature_name)); sym2ens={s:e for e,s in ens2sym.items()}
print('gene id map:',len(ens2sym))


In [ ]:
# PRIMARY: Geneformer pretrained gene embeddings -> cosine neighbors (functional proximity).
# We DON'T import the geneformer package (its collator breaks on new transformers); we only need
# the model's embedding matrix (AutoModel) + the token dictionary (pulled straight from the HF repo).
try:
    import torch, numpy as np, pickle
    from transformers import AutoModel
    from huggingface_hub import hf_hub_download, list_repo_files
    model=AutoModel.from_pretrained('ctheodoris/Geneformer')
    E=torch.nn.functional.normalize(model.get_input_embeddings().weight.detach().float(),dim=1).cpu().numpy()
    V=E.shape[0]; print('Geneformer embedding matrix:',E.shape)
    cands=[f for f in list_repo_files('ctheodoris/Geneformer') if 'token_dictionary' in f and f.endswith('.pkl')]
    g2t=None
    for f in sorted(cands):
        d=pickle.load(open(hf_hub_download('ctheodoris/Geneformer',f),'rb'))
        mx=max(v for v in d.values() if isinstance(v,int))
        if mx<V: g2t=d; print('token dict:',f,'(max token',mx,'< vocab',V,')'); break
    assert g2t is not None,'no token dictionary matches the model vocab'
    tok2ens={v:k for k,v in g2t.items()}
    for sym in TARGETS:
        ens=sym2ens.get(sym); tok=g2t.get(ens) if ens else None
        if tok is None or tok>=V: continue
        order=np.argsort(-(E@E[tok])); ds=[]
        for t in order[1:400]:
            s=ens2sym.get(tok2ens.get(int(t),''))
            if s and s!=sym: ds.append(s)
            if len(ds)>=30: break
        if ds: gf_out[sym]=ds
    gf_src='embedding-neighbors'
    print('Model 3 (embeddings): downstream sets for',len(gf_out),'regulators')
except Exception as e:
    import traceback; traceback.print_exc(); print('Geneformer embeddings skipped:',repr(e)[:200])
# NOTE: embedding cosine-neighbors were VALIDATED as ~random (0.8x baseline) for recovering real
# targets — Geneformer static embeddings are weak (same as the 0.53 essentiality result). So we do
# NOT feed these into the knockout cascade; they are saved for reference only. Real signal needs the
# InSilicoPerturber (RUN_ISP=True below). Cascade uses the measured graph unless ISP writes gf_perturb.json.
if gf_out: json.dump(gf_out,open(f'{OUT}/gf_embedding_neighbors.json','w'))  # reference only, not used by the cell
print('gf_perturb.json written:',bool(gf_out),'| source:',gf_src)


### (optional upgrade) true in-silico *delete* with InSilicoPerturber
Heavier and version-sensitive (needs the tokenized dataset + GPU). If it succeeds it **overwrites** `gf_perturb.json` with genuine deletion-shift downstream genes. Leave `RUN_ISP=False` to skip.


In [ ]:
RUN_ISP=False   # set True on a GPU runtime to run the real deletion perturbation
if RUN_ISP:
  # geneformer needs a transformers it is compatible with; install a pinned pair first
  !pip -q install 'transformers==4.37.2' 'geneformer' 2>/dev/null || echo 'pin install skipped'
  try:
    import scipy.sparse as sp
    from geneformer import TranscriptomeTokenizer, InSilicoPerturber, InSilicoPerturberStats
    assert adata is not None, 'need Model-2 adata'
    a=adata.copy(); a.var['ensembl_id']=a.var['feature_id']
    Xr=a.X; a.obs['n_counts']=np.asarray(Xr.sum(1)).ravel() if sp.issparse(Xr) else a.X.sum(1)
    os.makedirs('gf_in',exist_ok=True); a.write('gf_in/atlas.h5ad')
    TranscriptomeTokenizer({'cell_type':'cell_type'}).tokenize_data('gf_in','gf_tok','atlas',file_format='h5ad')
    ens_targets=[sym2ens[s] for s in TARGETS if s in sym2ens]
    isp=InSilicoPerturber(perturb_type='delete',genes_to_perturb=ens_targets,model_type='Pretrained',
         emb_mode='cell_and_gene',max_ncells=1000,forward_batch_size=32,nproc=4)
    isp.perturb_data('ctheodoris/Geneformer','gf_tok/atlas.dataset','gf_pert','pert')
    st=InSilicoPerturberStats(mode='aggregate_data'); st.get_stats('gf_pert',None,'gf_stats','stats')
    dfp=pd.read_csv('gf_stats/stats.csv'); print('ISP stats cols:',list(dfp.columns)[:8])
    gcol=next((c for c in dfp.columns if c.endswith('Gene_name') or c=='Gene_name'),None)
    acol=next((c for c in dfp.columns if 'Affected' in c and 'name' in c.lower()),None)
    scol=next((c for c in dfp.columns if 'shift' in c.lower() or 'Cosine' in c),dfp.columns[-1])
    if gcol and acol:
        real={}
        for g,grp in dfp.groupby(gcol):
            top=grp.sort_values(scol,ascending=False)[acol].head(30).tolist()
            s=ens2sym.get(str(g),str(g)); real[s]=[ens2sym.get(str(x),str(x)) for x in top]
        json.dump(real,open(f'{OUT}/gf_perturb.json','w')); print('ISP overwrote gf_perturb.json:',len(real))
  except Exception as e:
    import traceback; traceback.print_exc(); print('InSilicoPerturber failed, keeping embedding neighbors:',repr(e)[:200])


## Assemble → build → serve
The builder now folds in whatever the three models produced (present files override the fallbacks).


In [ ]:
# build with hard failure if anything is missing (so we never silently ship the old committed HTML)
import subprocess, time
if os.path.exists(f'{OUT}/cell_complete.json'): os.remove(f'{OUT}/cell_complete.json')
# builds all THREE cell apps: analysis cell, premium explorer, metabolic-routes.
# Model 4 needs the assembled features + the Perturb-seq h5ad, and its output feeds back in,
# so build_cell_complete runs once to make features, then compute_model4, then build again to fold it in.
for s in ['build_cell_complete.py','compute_model4.py','build_cell_complete.py','build_cell_app_complete.py','build_cell_explorer.py','compute_metabolic_graph.py','build_metabolic_routes.py']:
    r=subprocess.run([sys.executable,f'colab/{s}'],capture_output=True,text=True)
    print(r.stdout[-800:])
    if r.returncode!=0:
        print('BUILD ERROR in',s,'\n',r.stderr[-2500:]); raise SystemExit('build failed — fix the error above')
assert os.path.exists(f'{OUT}/cell_complete.json'), 'cell_complete.json was not produced'
# Phase 2 — convergence engine: novel functional links where independent lenses agree (reads the final model)
import os as _os; _os.environ.setdefault('CONV_PUBMED','1')   # rank top novel pairs by PubMed novelty
rc=subprocess.run([sys.executable,'colab/compute_convergence.py'],capture_output=True,text=True); print(rc.stdout[-1500:])
# Reasoning engine: derive new facts by logical composition (signed paths, transitive closure) + flag gaps
rr=subprocess.run([sys.executable,'colab/compute_reasoning.py'],capture_output=True,text=True); print(rr.stdout[-1200:])
# Tissue bridge (T1): instantiate the cell model per cell type from Model-2 expression
rt=subprocess.run([sys.executable,'colab/build_tissue_from_cells.py'],capture_output=True,text=True); print(rt.stdout[-800:])
# Transformer Tower A: assemble the typed KG + test link-prediction (is the KG predictive?)
rk=subprocess.run([sys.executable,'colab/build_kg_edges.py'],capture_output=True,text=True); print(rk.stdout[-600:])
ra=subprocess.run([sys.executable,'colab/train_tower_a.py'],capture_output=True,text=True); print(ra.stdout[-700:])
if os.environ.get('TRAIN_GNN','0')=='1':   # optional production GraphSAGE (GPU)
    !pip -q install torch
    rg=subprocess.run([sys.executable,'colab/train_tower_a_gnn.py'],capture_output=True,text=True); print(rg.stdout[-900:])
for f in ['cell_complete.html','cell_explorer.html','metabolic_routes.html']:
    print('FRESH',f+':', os.path.getsize(f'{OUT}/{f}')//1024,'KB')
if PROJ:
    import shutil
    for f in ['cell_complete.html','cell_explorer.html','metabolic_routes.html','cell_complete.json','predicted_essentiality.csv','depmap_essentiality.csv','celltype_masters.json','gf_perturb.json','convergence.json','reasoning.json','coexpr_neighbors.json','causal_reg.json','cell_instances.json','kg_edges.tsv']:
        p=f'{OUT}/{f}'
        if os.path.exists(p): shutil.copy(p,f'{PROJ}/{f}')
    print('copied all 3 apps + data to Drive:', PROJ)


In [ ]:
# [OUTPUT] bundle the WHOLE model into a single JSON you can download and hand back for evaluation
import json, gzip
D=json.load(open(f'{OUT}/cell_complete.json'))
def _n(x):
    try: return len(x)
    except: return x
# a compact SUMMARY (stats + small samples of every layer) — tiny, easy to upload to a chat
cel=D.get('celltypes')
summary={
 'manifest':{k:_n(v) for k,v in D.items()},
 'n_genes':len(D['genes']),
 'n_reg_edges':len(D.get('reg',[])), 'n_ppi_edges':len(D.get('ppi',[])),
 'dark_count':D.get('dark_count'), 'n_dark_predicted':len(D.get('darkfn',{})),
 'sample_genes':D['genes'][:5],
 'dark_predictions_sample':{k:D['darkfn'][k] for k in list(D.get('darkfn',{}))[:25]},
 'ti_priority_top':D.get('ti_priority',[])[:15],
 'ti_whitespace_top':D.get('ti_whitespace',[])[:15],
 'hiv_weakpoints':D.get('hiv_weakpoints',[])[:20],
 'celltypes':(list(cel.keys()) if isinstance(cel,dict) else cel),
}
json.dump(summary, open(f'{OUT}/cell_model_summary.json','w'), indent=1)
# gzip the FULL model (~13 MB -> ~3 MB) for a smaller upload
with open(f'{OUT}/cell_complete.json','rb') as fi, gzip.open(f'{OUT}/cell_complete.json.gz','wb') as fo: fo.writelines(fi)
print('full model :', os.path.getsize(f'{OUT}/cell_complete.json')//1024,'KB  ->  cell_complete.json')
print('gzipped    :', os.path.getsize(f'{OUT}/cell_complete.json.gz')//1024,'KB  ->  cell_complete.json.gz   (upload THIS — has everything)')
print('summary    :', os.path.getsize(f'{OUT}/cell_model_summary.json')//1024,'KB  ->  cell_model_summary.json   (tiny, upload if the gz is too big)')
print('layers in the model:', ', '.join(sorted(D.keys())))
if PROJ:
    import shutil
    for f in ['cell_complete.json.gz','cell_model_summary.json']: shutil.copy(f'{OUT}/{f}', f'{PROJ}/{f}')
    print('also saved both to Drive:', PROJ)
# also bundle the analysis deliverables (novel links, reasoning, KG) into one archive
import tarfile
with tarfile.open(f'{OUT}/analysis_outputs.tgz','w:gz') as tar:
    for f in ['convergence.json','reasoning.json','coexpr_neighbors.json','causal_reg.json','cell_instances.json','ncrna_targets.json','cell_model_summary.json']:
        if os.path.exists(f'{OUT}/{f}'): tar.add(f'{OUT}/{f}', arcname=f)
print('analysis   :', os.path.getsize(f'{OUT}/analysis_outputs.tgz')//1024,'KB  ->  analysis_outputs.tgz   (convergence + reasoning + lenses)')
if PROJ:
    import shutil; shutil.copy(f'{OUT}/analysis_outputs.tgz', f'{PROJ}/analysis_outputs.tgz')
if IN_COLAB:
    from google.colab import files
    try: files.download(f'{OUT}/cell_complete.json.gz'); files.download(f'{OUT}/cell_model_summary.json'); files.download(f'{OUT}/analysis_outputs.tgz')
    except Exception as e: print('auto-download skipped (', e, ') — grab the files from the file browser or Drive')


In [ ]:
# render inline (Colab) or print the localhost command (local)
if IN_COLAB:
    from IPython.display import HTML, display
    html=open(f'{OUT}/cell_complete.html').read().replace('"','&#34;')
    display(HTML(f'<iframe srcdoc="{html}" width=100% height=780 style=border:0></iframe>'))
else:
    print('run:  python colab/serve_cell.py   ->  http://localhost:8000/cell')


## What you can now do — with all three models live
- **Explore** → click any protein: full trafficking journey + networks; essentiality tagged *measured* vs *our model* (Model 1).
- **cell type** dropdown → data-driven master-TF networks from the atlas (Model 2).
- **Remove/Mutate** → cascade over the **measured** reg+PPI graph (validated). Geneformer embedding-neighbors were checked and are ~random, so they are NOT fed into the cascade; run RUN_ISP=True for real in-silico perturbation.
- **Dark genes** → the function frontier now carries a **predicted function** per gene: guilt-by-association from *measured* functional neighbors (Perturb-seq perturbation-response similarity + co-essentiality + physical interactions), each tagged *predicted, not known*.
- **Predicted perturbation response (Model 4)** → click a never-perturbed gene: its predicted KO-signature neighbors + affected pathway, with the **held-out recall-vs-random lift** shown so you know how much to trust it (honest by construction — a ~1x lift is reported as noise).
- **Metabolism / Infect: HIV** → reactions, and HIV's weak points.
